> Copyright (c) 2022 DFKI GmbH - All Rights Reserved
> 
> Written by Michael Fürst <Michael.Fuerst@dfki.de>, October 2022

# Example: NuScenes - Create Tars

Please read the documentation on converting datasets before following this example. (See in doc "datapipes.dataset_converter")

This dataset and tutorial was created with the following versions.

In [ ]:
from datapipes.versions import api_version, metadata_file_format_version

print(f"API-Version: {api_version}")
print(f"File Format Version: {metadata_file_format_version}")

## **Step 1**: Gather information on the dataset

The original nuScenes dataset has all annotations in a single annotation database which is indexed using tokens (sample token and sensor token), the images and pointclouds are then stored separately in a file for each. Filenames can be retrieved from the annotation database if the token is known. However, you cannot infer the token given the filename.


### Extract FileInfo

Due to this structure, we iterate over the dataset and collect all FileInfo for the files that way, instead of parsing it from the filenames.
With that list we can use the build_dataset_structure function from the library and continue using pre-implemented functions.

In [ ]:
import os
from typing import Dict, List
from tqdm import tqdm
from os.path import join
from nuscenes import NuScenes
from nuscenes.utils.splits import mini_train, mini_val

from datapipes.dataset_converter import FileInfo

RAW_DATASET="/ds-av/public_datasets/nuscenes/raw"
CAMERA_SENSORS = [
    "CAM_FRONT","CAM_FRONT_RIGHT", "CAM_FRONT_LEFT",
    "CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT"
]
LIDAR_SENSORS = ["LIDAR_TOP"]
RADAR_SENSORS = [
    "RADAR_FRONT", "RADAR_FRONT_RIGHT", "RADAR_FRONT_LEFT",
    "RADAR_BACK_LEFT", "RADAR_BACK_RIGHT"
]


def gather_fileinfos(dataroot, subsets) -> List[FileInfo]:
    """
    Get a list of sample info.
    """
    file_infos = []
    for subset_name, version, sequences in subsets:
        dataset = NuScenes(version=version, dataroot=dataroot, verbose=False)
    
        for scene in tqdm(dataset.scene, desc="Collecting tokens"):
            if scene["name"] not in sequences: continue
            sample_token = scene['first_sample_token']
            sample = dataset.get('sample', sample_token)
            file_infos.extend(
                collect_fileinfo(dataroot, subset_name, dataset, scene, sample_token, sample)
            )
            while sample_token != scene['last_sample_token']:
                sample_token = sample["next"]
                sample = dataset.get('sample', sample_token)
                file_infos.extend(
                    collect_fileinfo(dataroot, subset_name, dataset, scene, sample_token, sample)
                )
    return file_infos

def collect_fileinfo(dataroot, subset_name, dataset, scene, sample_token, sample):
    file_infos = []
    for sensor in CAMERA_SENSORS + LIDAR_SENSORS + RADAR_SENSORS:
        sample_data = dataset.get('sample_data', sample['data'][sensor])
        file_path = join(dataroot, sample_data["filename"])
        file_infos.append(FileInfo(
            subset=subset_name,
            sequence_name=scene["name"],
            sample_name=sample_token,
            component_id=sensor,
            file_path=file_path,
            file_size=os.path.getsize(file_path)
        ))
    return file_infos

file_infos = gather_fileinfos(RAW_DATASET, [
    ("mini_train", "v1.0-mini", mini_train),
    ("mini_val", "v1.0-mini", mini_val),
])

for info in file_infos[:2]:
    print(info)
print(f"(output truncated after 10/{len(file_infos)} entries)")

### Use Cases

The most common usage for nuScenes is to use all keyframes, but not all sensors. Common sensor configurations in use are:

* Lidar only
* Radar (all) only
* Camera (all) only
* Camera front only
* Lidar + Camera
* Radar + Camera
* any other combination

### Packaging Separately

Due to the size of the data and based on these use cases, we want to split data into separate TARs.

1. LiDAR is in a separate tar.
2. Cameras are in a separate tar per camera.
3. Radar is in a separate tar, but all Radars are in a single tar (they are small pointclouds).
4. Annotations are not packaged at all, since they are in a single Database stored as a JSON file already.

### Build dataset structure

Since we created the list of fileinfos it is very simple to build the dataset structure. We simply call a function from the datapipes library and define a component map. The component map defines how sensors are mapped into groups of components stored in a single tar.

In [ ]:
from datapipes.dataset_converter import build_dataset_structure

COMPONENT_MAP = dict(
    # The lidar goes in a tar
    lidar=["LIDAR_TOP"],
    # All radars go in a single tar
    radar=RADAR_SENSORS,
    # Put each camera sensor in a separate tar
    **{ x.lower(): [x] for x in CAMERA_SENSORS}
)

dataset_structure = build_dataset_structure(file_infos, COMPONENT_MAP)

# Print a sample
print(list(dataset_structure["mini_train"]["scene-0061"]["scene-0061.ca9a282c9e77460f8360f564131a8af5"].keys()))
print(dataset_structure["mini_train"]["scene-0061"]["scene-0061.ca9a282c9e77460f8360f564131a8af5"]["radar"])

## **Step 2**: Shard the dataset

First let's figure out the shard sizes. For this we can use the suggestion tool and see if the results make sense.

In [ ]:
from datapipes.dataset_converter import suggest_shard_size

shard_sizes = suggest_shard_size(dataset_structure)

Now that we have the size suggestions and inspected it, we actually assign the files to the shards.

For sharding we will do it different for "train" and "val"/"test".
The latter should remain sorted, so that trackers could be applied, but for training, we want as much randomness as possible.

This means on "train" we will merge all sequences in a single sequence and shuffle before sharding. On the other hand for "val" and "test" we will create the shards while preserving the sequence boundaries.

In [ ]:
from datapipes.dataset_converter import assign_files_to_shards

shard_info = assign_files_to_shards(
    dataset_structure,
    shard_sizes,
    preserve_sequence_boundaries_for_subsets=["mini_val"],
    global_shuffle_for_subsets=["mini_train"]
)

print(f"Shards (mini_train): {list(shard_info['mini_train'].keys())}")
print(f"Shards (mini_val): {list(shard_info['mini_val'].keys())}")

## **Step 3**: Write Shards and Metainfo

Finally with the shardinfo we can write the shards and metainfo so we can load them again.

In [ ]:
from datapipes.dataset_converter import write_metadata, write_shards

TAR_DATASET = "/home/fuerst/Desktop/Datasets/nuscenes/keyframes_mini"

metadata = write_metadata(shard_info, TAR_DATASET, subsets_to_pre_shuffle=["train"])
write_shards(metadata, shard_info, RAW_DATASET, TAR_DATASET, overwrite=True)

We also need to copy over the original annotation database.
Since these are monolithic files anyways and we want to use the original loader we do not put them in tars. A simple copy is enough.

In [ ]:
import os
import shutil

def copy_annotation_databases(dataset_path, target_path, version):
    database_files = [
        "attribute.json",
        "calibrated_sensor.json",
        "category.json",
        "ego_pose.json",
        "instance.json",
        "lidarseg.json",
        "log.json",
        "map.json",
        "panoptic.json",
        "sample.json",
        "sample_annotation.json",
        "sample_data.json",
        "scene.json",
        "sensor.json",
        "visibility.json"
    ]
    os.makedirs(os.path.join (target_path, version), exist_ok=True)
    for name in database_files:
        shutil.copyfile(os.path.join(dataset_path, version, name), os.path.join (target_path, version, name))

copy_annotation_databases(RAW_DATASET, TAR_DATASET, version="v1.0-mini")

## **Step 4**: Validate the tars

We list and inspect all the tars. For speed we will only use the mini dataset for testing.

To do this we first list all tars that match and then print first N filenames in the tar to check if the order matches.

In [ ]:
from os import listdir
from os.path import join
import tarfile

def check_tars(tar_folder, split, mini=False):
    tars = listdir(tar_folder)
    def _is_tar(fname):
        return fname.endswith(".tar")
    tars = filter(_is_tar, tars)
    if mini:
        tars = filter(lambda fname: "mini" in fname, tars)
    else:
        tars = filter(lambda fname: not "mini" in fname, tars)
    tars = filter(lambda fname: split in fname, tars)
    tars = sorted(tars)
    for tar_idx, fname in enumerate(tars):
        print(fname)
        if tar_idx == 0:
            with tarfile.open(join(tar_folder, fname), mode="r") as tar:
                for idx, info in enumerate(tar):
                    print(f"   {info.name}")
                    if idx >= 2:             
                        print("   ...")
                        break
        else:
            print("   ...")


check_tars(tar_folder=TAR_DATASET, split="train", mini=True)

## **Step 5**: Redo for full dataset

So far we have only processed nuScenes mini for testing purposes. Now it is time to process the entire nuScenes dataset.

We do this in two parts: First we create the shard assignment and inspect it; then we write it to disk.

In [ ]:
from nuscenes.utils.splits import train, val, test

file_infos = gather_fileinfos(RAW_DATASET, [
    ("train", "v1.0-trainval", train),
    ("val", "v1.0-trainval", val),
    ("test", "v1.0-test", test),
])
dataset_structure = build_dataset_structure(file_infos, COMPONENT_MAP)
shard_sizes = suggest_shard_size(dataset_structure)
shard_info = assign_files_to_shards(
    dataset_structure,
    shard_sizes,
    global_shuffle_for_subsets=["train"],
    preserve_sequence_boundaries_for_subsets=["val", "test"]
)
for subset, shards in shard_info.items():
    print(f"Shards ({subset}): {list(shards.keys())}")

Looks good. Let's write it.

In [ ]:
TAR_DATASET = "/home/fuerst/Desktop/Datasets/nuscenes/keyframes"
metadata = write_metadata(shard_info, TAR_DATASET, subsets_to_pre_shuffle=["train"])
write_shards(metadata, shard_info, RAW_DATASET, TAR_DATASET)

And copy all the annotation databases.

In [ ]:
copy_annotation_databases(RAW_DATASET, TAR_DATASET, version="v1.0-trainval")
copy_annotation_databases(RAW_DATASET, TAR_DATASET, version="v1.0-test")

## **Step 6**: Create a file with dataset info

Finally, to show up correctly on the dataset overview, we have to create a file with an overview over the dataset.
To create that file we use the write_ds_info file.

Note that the file is only created once for all variants (mini and full), since these properties are shared and not specific to the conversion done here.

In [ ]:
from datapipes.dataset_converter import encode_ds_info, write_ds_info

ds_info = encode_ds_info(
    short_name="nuscenes",
    full_name="nuScenes - Detection",
    sensors=["RGB", "radar", "lidar"],
    camera_setup="other",
    nature_of_data="real",
    tasks=["3d object detection"],
    project_page="https://www.nuscenes.org/nuscenes",
    code_repo="https://github.com/nutonomy/nuscenes-devkit",
    paper_url="https://openaccess.thecvf.com/content_CVPR_2020/papers/Caesar_nuScenes_A_Multimodal_Dataset_for_Autonomous_Driving_CVPR_2020_paper.pdf",
    license_name="CC BY-NC-SA 4.0",
    converted_by=["Michael Fuerst"],
)
write_ds_info(
    ds_info,
    output_dir="/home/fuerst/Desktop/Datasets/nuscenes"
)